# 02. CRDNN — обучение char-vocab ASR

Self-contained ноутбук: от сырых данных до submission.
Архитектура: CRDNN (Conv2D frontend + BiGRU + CTC head, ~3.6M параметров).
HP Random Search оборачивает весь цикл обучения.

## Установка зависимостей

Выполнять под свою платформу — локально обычно уже установлено через `uv sync`; на Colab/Kaggle — раскомментируй нужный блок.

In [1]:
# Idempotent data download. Skip if notebooks/data/ already populated.
from pathlib import Path
import zipfile

DATA_ROOT = Path("data")
_ZIP_URL = "https://drive.google.com/file/d/1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW/view?usp=share_link"
_ZIP_PATH = DATA_ROOT / "data.zip"

if not DATA_ROOT.exists() or not any(DATA_ROOT.iterdir()):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    import gdown

    gdown.download(url=_ZIP_URL, output=str(_ZIP_PATH), quiet=False, fuzzy=True)
    with zipfile.ZipFile(_ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT)
    print(f"Extracted data to {DATA_ROOT}")
else:
    print(f"Data already present at {DATA_ROOT} — skipping download.")


Data already present at data — skipping download.


## Env hardening

In [ ]:
# Env hardening — must run BEFORE `import torch`.
import os

# Снижает фрагментацию VRAM — критично для torch.compile + переменных T.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

import torch

# cudnn.benchmark=False — переменные T после padding, autotune только мешает.
torch.backends.cudnn.benchmark = False
# TF32 для matmul — ускоряет fp32 операции на Ampere+ без потери CER.
torch.set_float32_matmul_precision("high")

# Limit torch.compile graph cache (dynamic=True can cache 50+ variants).
import torch._dynamo
torch._dynamo.config.cache_size_limit = 8


## Пути (заполните вручную)

Задай абсолютные пути под свою среду. Все `FILL_ME_IN` должны быть реальными путями к train/dev/test и директории чекпоинтов.

In [3]:
from pathlib import Path

# DATA_ROOT was defined in the download cell above.
TRAIN_ROOT = DATA_ROOT / "data" / "train"
DEV_ROOT = DATA_ROOT / "data" / "dev"
TEST_ROOT: Path | None = DATA_ROOT / "data" / "test"
TRAIN_CSV = TRAIN_ROOT / "train.csv"
DEV_CSV = DEV_ROOT / "dev.csv"
CKPT_ROOT = Path("../checkpoints") / "02_crdnn"

for p in (TRAIN_ROOT, DEV_ROOT, TRAIN_CSV, DEV_CSV):
    assert p.exists(), f"Path does not exist: {p}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"train={TRAIN_ROOT}, dev={DEV_ROOT}, ckpts={CKPT_ROOT}")

MUSAN_ROOT = Path.home() / "datasets" / "musan" / "noise"
RIR_ROOT   = Path.home() / "datasets" / "RIRS_NOISES" / "simulated_rirs"

assert MUSAN_ROOT.exists(), f"musan not found: {MUSAN_ROOT}"
assert RIR_ROOT.exists(),   f"RIRs not found:  {RIR_ROOT}"

train=data/data/train, dev=data/data/dev, ckpts=../checkpoints/02_crdnn


## Шаг 1. Импорты

In [4]:
import json
import random
import time

import torch
from torch.utils.data import DataLoader

from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.audio_aug_gpu import GPUAudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.data.manifest import records_from_csv
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.crdnn import CRDNN
from gp1.text.vocab import CharVocab
from gp1.text.denormalize import words_to_digits
from gp1.train.trainer import Trainer, TrainerConfig
from gp1.train.optim import build_adamw
from gp1.train.schedulers import build_cosine_warmup
from gp1.train.checkpoint import load_checkpoint
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.types import AugConfig

# device must be constructed AFTER `import torch`.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

device=cuda


## Шаг 2. Гиперпараметры (FIXED + HP_GRID)

In [5]:
FIXED = {
    "samplerate": 16000,
    "n_fft": 512,
    "n_mels": 80,
    "hop_length": 160,
    "win_length": 400,
    "max_epochs": 50,
    "grad_accum": 1,
    "subsample_factor": 1,
}

HP_GRID = {
    
    # --- model ---
    "d_cnn":       [32],
    "rnn_hidden":  [192, 256],
    "rnn_layers":  [2, 3],

    # --- optimizer / schedule ---
    "lr":                        [0.010, 0.015, 0.020],
    "weight_decay":              [1e-4, 1e-3],
    "dropout":                   [0.10, 0.15, 0.20],
    "warmup_steps":              [200, 500, 1000],
    "min_lr_ratio":              [0.01, 0.05],

    # --- SpecAugment (spectrogram) ---
    "specaug_freq_mask_param":   [10, 15, 20],
    "specaug_num_freq_masks":    [1, 2],
    "specaug_time_mask_param":   [20, 25, 35],
    "specaug_num_time_masks":    [2, 5, 8],
    "specaug_time_mask_max_ratio": [0.04, 0.05, 0.08],

    # --- Audio: speed & pitch (XOR) ---
    "speed_prob":                [0.5, 1.0],
    "speed_factors":             [(0.9, 1.0, 1.1), (0.85, 1.0, 1.15)],
    "pitch_prob":                [0.0, 0.3, 0.5],
    "pitch_range_semitones":     [(-2.0, 2.0), (-3.0, 3.0), (-4.0, 4.0)],

    # --- Audio: gain / VTLP / noise / RIR ---
    "gain_prob":                 [0.3, 0.7],
    "gain_db_range":             [(-4.0, 4.0), (-8.0, 8.0)],
    "vtlp_prob":                 [0.0, 0.3, 0.5],
    "vtlp_alpha_range":          [(0.9, 1.1), (0.85, 1.15)],
    "noise_prob":                [0.0, 0.3],
    "noise_snr_db_range":        [(10.0, 20.0), (5.0, 20.0)],
    "rir_prob":                  [0.0, 0.1]
}

N_TRIALS = 5
SEED = 42

print("FIXED:", FIXED)
print("N_TRIALS:", N_TRIALS)


FIXED: {'samplerate': 16000, 'n_fft': 512, 'n_mels': 80, 'hop_length': 160, 'win_length': 400, 'max_epochs': 50, 'grad_accum': 1, 'subsample_factor': 1}
N_TRIALS: 5


## Шаг 3. Загрузка записей из CSV

In [6]:
train_records = records_from_csv(TRAIN_CSV, TRAIN_ROOT)
dev_records = records_from_csv(DEV_CSV, DEV_ROOT)
print(f"Train records: {len(train_records)}, Dev records: {len(dev_records)}")

in_domain_speakers = {r.spk_id for r in train_records}
out_of_domain_count = sum(1 for r in dev_records if r.spk_id not in in_domain_speakers)
in_domain_count = sum(1 for r in dev_records if r.spk_id in in_domain_speakers)
print(f"dev speakers: in-domain={in_domain_count} samples, out-of-domain={out_of_domain_count} samples")


Train records: 12553, Dev records: 2265
dev speakers: in-domain=600 samples, out-of-domain=1665 samples


## Шаг 4. Vocabulary

In [7]:
vocab = CharVocab()
print(f"Vocab size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")


Vocab size: 35, blank_id: 0


## Шаг 4.5. Предзагрузка аудио в RAM (опционально, сильно ускоряет обучение)

Загружает все train+dev файлы один раз, применяет resample до 16 kHz, и держит в RAM как тензоры. После этого `SpokenNumbersDataset.__getitem__` пропускает `sf.read` + `Resample` полностью.

Стоит ~3-5 минут один раз. Нужно ~2 GB RAM для GP1 (Colab: 12 GB, Kaggle: 29 GB — влезает с запасом).

In [8]:
from gp1.data.dataset import preload_audio_cache

AUDIO_CACHE = preload_audio_cache(
    train_records + dev_records,
    target_samplerate=FIXED["samplerate"],
)

for k in list(AUDIO_CACHE.keys()):
    AUDIO_CACHE[k] = AUDIO_CACHE[k].contiguous().share_memory_()


preload audio: 100%|██████████| 14818/14818 [00:13<00:00, 1114.74it/s]


## Шаг 5. HP Random Search — Training Loop

In [9]:
import os
import random

import torch


def _worker_init(worker_id: int) -> None:
    """1 BLAS-thread per worker + per-worker RNG seed for augmenter."""
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    torch.set_num_threads(1)

    info = torch.utils.data.get_worker_info()
    if info is None:
        return
    aug = getattr(info.dataset, "_augmenter", None)
    if aug is not None and hasattr(aug, "_rng"):
        aug._rng = random.Random(info.seed & 0xFFFFFFFF)


In [10]:
import math
import gc

BATCH_SIZE = 96
DL_WORKERS = 8

SEED = 42
random.seed(SEED)
trial_results = []
run_root = CKPT_ROOT / f"02_crdnn_{int(time.time())}"
run_root.mkdir(parents=True, exist_ok=True)


for trial in range(N_TRIALS):
    hp = {k: random.choice(v) for k, v in HP_GRID.items()}
    print(f"\n=== Trial {trial + 1}/{N_TRIALS} ===")
    print(json.dumps({**FIXED, **hp}, default=str, indent=2))

    aug_cfg = AugConfig(
        speed_factors=tuple(hp["speed_factors"]),
        speed_prob=hp["speed_prob"],
        pitch_prob=hp["pitch_prob"],
        pitch_range_semitones=tuple(hp["pitch_range_semitones"]),
        gain_prob=hp["gain_prob"],
        gain_db_range=tuple(hp["gain_db_range"]),
        seed=SEED + trial,
    )
    train_aug = AudioAugmenter(aug_cfg)
    gpu_aug = GPUAudioAugmenter(
        samplerate=FIXED["samplerate"],
        vtlp_prob=hp["vtlp_prob"],
        vtlp_alpha_range=tuple(hp["vtlp_alpha_range"]),
        noise_prob=hp["noise_prob"],
        noise_snr_db_range=tuple(hp["noise_snr_db_range"]),
        musan_root=MUSAN_ROOT,
        rir_prob=hp["rir_prob"],
        rir_root=RIR_ROOT,
    )
    spec_aug = SpecAugmenter(
        freq_mask_param=hp["specaug_freq_mask_param"],
        num_freq_masks=hp["specaug_num_freq_masks"],
        time_mask_param=hp["specaug_time_mask_param"],
        num_time_masks=hp["specaug_num_time_masks"],
        time_mask_max_ratio=hp["specaug_time_mask_max_ratio"],
        seed=SEED + trial,
    )

    train_ds = SpokenNumbersDataset(
        records=train_records,
        vocab=vocab,
        augmenter=train_aug,
        target_samplerate=FIXED["samplerate"],
        audio_cache=AUDIO_CACHE,
    )
    dev_ds = SpokenNumbersDataset(
        records=dev_records,
        vocab=vocab,
        augmenter=None,
        target_samplerate=FIXED["samplerate"],
        audio_cache=AUDIO_CACHE,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=DL_WORKERS,
        persistent_workers=True,
        pin_memory=True,
        prefetch_factor=3,
        worker_init_fn=_worker_init,
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=DL_WORKERS,
        persistent_workers=True,
        pin_memory=True,
        prefetch_factor=3,
        worker_init_fn=_worker_init,
    )

    model = CRDNN(
        vocab_size=vocab.vocab_size,
        d_cnn=hp["d_cnn"],
        rnn_hidden=hp["rnn_hidden"],
        rnn_layers=hp["rnn_layers"],
        dropout=hp["dropout"],
        subsample_factor=FIXED["subsample_factor"],
    ).to(device)
    model = torch.compile(model, mode="default", dynamic=True)

    ctc = CTCLoss(blank_id=vocab.blank_id)
    optimizer = build_adamw(
        model.parameters(),
        lr=hp["lr"],
        weight_decay=hp["weight_decay"],
    )
    steps_per_epoch = math.ceil(len(train_loader) / FIXED["grad_accum"])
    total_steps = steps_per_epoch * FIXED["max_epochs"]
    scheduler = build_cosine_warmup(
        optimizer,
        total_steps=total_steps,
        warmup_steps=hp["warmup_steps"],
        min_lr_ratio=hp["min_lr_ratio"],
    )

    trial_dir = run_root / f"trial_{trial:02d}"
    cfg = TrainerConfig(
        max_epochs=FIXED["max_epochs"],
        grad_accum=FIXED["grad_accum"],
        fp16_autocast=True,
        val_every_n_epochs=1,
        early_stop_patience=5,
        early_stop_metric="harmonic_in_out_cer",
        in_domain_speakers=in_domain_speakers,
        ckpt_dir=trial_dir,
    )
    trainer = Trainer(
        model=model,
        ctc_loss=ctc,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        val_loader=dev_loader,
        vocab=vocab,
        config=cfg,
        device=device,
        audio_cfg={
            "n_fft": FIXED["n_fft"],
            "n_mels": FIXED["n_mels"],
            "hop_length": FIXED["hop_length"],
            "win_length": FIXED["win_length"],
            "samplerate": FIXED["samplerate"],
        },
        spec_augmenter=spec_aug,
        gpu_augmenter=gpu_aug,
    )
    result = trainer.fit()
    best_ckpt = result["best_ckpt_path"]
    trial_results.append({"trial": trial, "hp": hp, **result})

    if torch.cuda.is_available():
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f"trial {trial}: peak VRAM = {peak_gb:.2f} GB")

    if best_ckpt is not None:
        with open(trial_dir / "meta.json", "w") as f:
            json.dump(
                {
                    "arch": "crdnn",
                    "hparams": {**FIXED, **hp},
                    "val_cer": result["best_monitored"],
                    "trial": trial,
                },
                f,
                indent=2,
                default=str,
            )
        # Cleanup в конце trial.
    del trainer, model, optimizer, scheduler, ctc
    del train_loader, dev_loader, train_ds, dev_ds
    del train_aug, gpu_aug, spec_aug
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    torch._dynamo.reset()

print("\nHP search complete.")


=== Trial 1/5 ===
{
  "samplerate": 16000,
  "n_fft": 512,
  "n_mels": 80,
  "hop_length": 160,
  "win_length": 400,
  "max_epochs": 50,
  "grad_accum": 1,
  "subsample_factor": 1,
  "d_cnn": 32,
  "rnn_hidden": 192,
  "rnn_layers": 3,
  "lr": 0.01,
  "weight_decay": 0.0001,
  "dropout": 0.1,
  "warmup_steps": 1000,
  "min_lr_ratio": 0.01,
  "specaug_freq_mask_param": 20,
  "specaug_num_freq_masks": 1,
  "specaug_time_mask_param": 35,
  "specaug_num_time_masks": 5,
  "specaug_time_mask_max_ratio": 0.04,
  "speed_prob": 0.5,
  "speed_factors": [
    0.9,
    1.0,
    1.1
  ],
  "pitch_prob": 0.0,
  "pitch_range_semitones": [
    -2.0,
    2.0
  ],
  "gain_prob": 0.3,
  "gain_db_range": [
    -4.0,
    4.0
  ],
  "vtlp_prob": 0.5,
  "vtlp_alpha_range": [
    0.85,
    1.15
  ],
  "noise_prob": 0.0,
  "noise_snr_db_range": [
    5.0,
    20.0
  ],
  "rir_prob": 0.1
}


epochs:   0%|          | 0/50 [00:00<?, ?it/s]

epoch 1:   0%|          | 0/131 [00:00<?, ?it/s]

[Epoch 1/50] train  | loss=4.2796
[Epoch 1/50] val    | loss=3.0150  hm_cer=0.7208  (in=0.7161  out=0.7257)  best=0.7208  no_improve=0/5


epoch 2:   0%|          | 0/131 [00:00<?, ?it/s]

...

[Epoch 7/50] train  | loss=1.6221
[Epoch 7/50] val    | loss=1.8647  hm_cer=0.5023  (in=0.4986  out=0.5061)  best=0.2967  no_improve=5/5
trial 2: peak VRAM = 8.43 GB

=== Trial 4/5 ===
{
  "samplerate": 16000,
  "n_fft": 512,
  "n_mels": 80,
  "hop_length": 160,
  "win_length": 400,
  "max_epochs": 50,
  "grad_accum": 1,
  "subsample_factor": 1,
  "d_cnn": 64,
  "rnn_hidden": 256,
  "rnn_layers": 2,
  "lr": 0.02,
  "weight_decay": 0.001,
  "dropout": 0.1,
  "warmup_steps": 200,
  "min_lr_ratio": 0.01,
  "specaug_freq_mask_param": 15,
  "specaug_num_freq_masks": 2,
  "specaug_time_mask_param": 25,
  "specaug_num_time_masks": 2,
  "specaug_time_mask_max_ratio": 0.04,
  "speed_prob": 1.0,
  "speed_factors": [
    0.9,
    1.0,
    1.1
  ],
  "pitch_prob": 0.5,
  "pitch_range_semitones": [
    -3.0,
    3.0
  ],
  "gain_prob": 0.7,
  "gain_db_range": [
    -8.0,
    8.0
  ],
  "vtlp_prob": 0.0,
  "vtlp_alpha_range": [
    0.85,
    1.15
  ],
  "noise_prob": 0.0,
  "noise_snr_db_range": [
 

epochs:   0%|          | 0/50 [00:00<?, ?it/s]

epoch 1:   0%|          | 0/131 [00:00<?, ?it/s]

W0425 01:41:50.936000 336260 torch/_inductor/utils.py:1613] [0/0_2] Not enough SMs to use max_autotune_gemm mode


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.63 GiB. GPU 0 has a total capacity of 11.48 GiB of which 1.37 GiB is free. Including non-PyTorch memory, this process has 9.31 GiB memory in use. Of the allocated memory 5.56 GiB is allocated by PyTorch, and 64.68 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Шаг 6. Сводный отчёт трайлов

In [ ]:
trial_results_sorted = sorted(trial_results, key=lambda r: r["best_monitored"])
print(f"{'trial':>5} {'best_val_cer':>12} {'lr':>8} {'dropout':>8} {'rnn_hidden':>10} {'rnn_layers':>10}")
print("-" * 60)
for r in trial_results_sorted:
    hp = r["hp"]
    print(
        f"{r['trial']:>5}"
        f" {r['best_monitored']:>12.4f}"
        f" {hp['lr']:>8.5f}"
        f" {hp['dropout']:>8.4f}"
        f" {hp['rnn_hidden']:>10}"
        f" {hp['rnn_layers']:>10}"
    )

## Шаг 7. Валидация лучшей модели на dev + greedy predictions

In [14]:
from gp1.text.normalize import digits_to_words

# best_result = trial_results_sorted[0]
# best_ckpt = best_result["best_ckpt_path"]
# best_hp = json.loads(best_result["meta"])["hparams"]
# print("Best trial val_cer:", best_result["best_monitored"], "ckpt:", best_ckpt)

with open("/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/checkpoints/02_crdnn/02_crdnn_1777066371/trial_00/meta.json", "r") as f:
    best_hp = json.load(f)["hparams"]
best_ckpt = Path("/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/checkpoints/02_crdnn/02_crdnn_1777066371/trial_00/best.pt")

model = CRDNN(
    vocab_size=vocab.vocab_size,
    d_cnn=best_hp["d_cnn"],
    rnn_hidden=best_hp["rnn_hidden"],
    rnn_layers=best_hp["rnn_layers"],
    dropout=best_hp["dropout"],
    subsample_factor=FIXED["subsample_factor"],
).to(device)
meta = load_checkpoint(best_ckpt, model)
model.eval()

mel_extractor = LogMelFilterBanks(
    n_fft=FIXED["n_fft"],
    samplerate=FIXED["samplerate"],
    hop_length=FIXED["hop_length"],
    win_length=FIXED["win_length"],
    n_mels=FIXED["n_mels"],
).to(device)

dev_ds_eval = SpokenNumbersDataset(
    records=dev_records, vocab=vocab, augmenter=None, target_samplerate=FIXED["samplerate"]
)
dev_loader_eval = DataLoader(
    dev_ds_eval, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=4,
        persistent_workers=True,
        pin_memory=True,
        prefetch_factor=4,
    )

predictions, refs, spks = [], [], []
with torch.no_grad():
    for batch in dev_loader_eval:
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        mel = mel_extractor(audio)
        mel_lengths = (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
        out = model(mel, mel_lengths)
        hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
        predictions.extend(hyps)
        refs.extend(digits_to_words(t) for t in batch.transcriptions)
        spks.extend(batch.spk_ids)

cer = compute_cer(refs, predictions)
per_spk = compute_per_speaker_cer(refs, predictions, spks)
print(f"Dev CER (greedy): {cer:.4f}")

Dev CER (greedy): 0.1685
